In [1]:
import sys
from pathlib import Path

# Add repo root to Python path
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

print("Repo root added to sys.path:", REPO_ROOT)

Repo root added to sys.path: c:\Users\jiaya\OneDrive\Documents\Lund_2025\Thesis\unified-probabilistic-validation


In [2]:
import numpy as np
import pandas as pd

from src.conformal.utils import coverage_and_width, weighted_quantile
from src.conformal.split_points import split_conformal_interval_point
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from src.conformal.online import (online_conformal_point,online_conformal_point_rolling)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add repo root to Python path so `import src...` works
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print("Repo root added to sys.path:", REPO_ROOT)

from src.conformal.utils import coverage_and_width, weighted_quantile
from src.conformal.split_points import split_conformal_interval_point
from src.conformal.online import online_conformal_point

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ----------------------------
# 0) Load LONG artifacts
# ----------------------------
DATA_DIR = REPO_ROOT / "data" / "derived_long"
print("Loading from:", DATA_DIR)

y = np.load(DATA_DIR / "entsog_y.npy")
yhat = np.load(DATA_DIR / "entsog_yhat.npy")
t = np.load(DATA_DIR / "entsog_t.npy", allow_pickle=True)
scale = np.load(DATA_DIR / "entsog_scale.npy")

assert len(y) == len(yhat) == len(t) == len(scale)
n = len(y)
print("Loaded shapes:", y.shape, yhat.shape, scale.shape)

# ----------------------------
# 1) Time split (cal = early, test = late)
# ----------------------------
split = int(0.7 * n)

y_cal, y_test = y[:split], y[split:]
yhat_cal, yhat_test = yhat[:split], yhat[split:]
t_cal, t_test = t[:split], t[split:]
scale_cal, scale_test = scale[:split], scale[split:]

print("cal n:", len(y_cal), "test n:", len(y_test))

# ----------------------------
# 2) Covariate-shift weights (optional)
# ----------------------------
def make_time_features(timestamps):
    ts = pd.to_datetime(timestamps)
    hour = ts.hour
    dow = ts.dayofweek
    return np.column_stack([
        np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24),
        np.sin(2*np.pi*dow/7),   np.cos(2*np.pi*dow/7)
    ])

def build_density_ratio_weights(X_cal, X_test):
    X = np.vstack([X_cal, X_test])
    lab = np.hstack([np.zeros(len(X_cal)), np.ones(len(X_test))])

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xs, lab)

    p_test = clf.predict_proba(scaler.transform(X_cal))[:, 1]
    p_cal = 1.0 - p_test

    eps = 1e-12
    odds = (p_test + eps) / (p_cal + eps)
    w = odds * (len(X_cal) / max(1, len(X_test)))
    return np.clip(w, 0.05, 20.0)

X_cal = make_time_features(t_cal)
X_test = make_time_features(t_test)
w_cal = build_density_ratio_weights(X_cal, X_test)
print("w_cal min/max:", float(w_cal.min()), float(w_cal.max()))

# toggle weighting here:
w_cal_local = w_cal   # or None

# ----------------------------
# 3) Run conformal experiments for alpha in {0.2, 0.1}
# ----------------------------
for alpha in [0.2, 0.1]:
    # load base intervals matching alpha
    if alpha == 0.2:
        lo_base = np.load(DATA_DIR / "entsog_lo_base_80.npy")
        hi_base = np.load(DATA_DIR / "entsog_hi_base_80.npy")
    else:
        lo_base = np.load(DATA_DIR / "entsog_lo_base_90.npy")
        hi_base = np.load(DATA_DIR / "entsog_hi_base_90.npy")

    assert len(lo_base) == len(y) == len(hi_base)

    lo_cal, lo_test = lo_base[:split], lo_base[split:]
    hi_cal, hi_test = hi_base[:split], hi_base[split:]

    print("\n" + "="*60)
    print(f"ALPHA={alpha}  (nominal coverage={1-alpha:.0%})")

    # --- Split conformal on point forecasts (constant-width) ---
    lo_p, hi_p, q_p = split_conformal_interval_point(
        y_cal=y_cal,
        yhat_cal=yhat_cal,
        yhat_test=yhat_test,
        alpha=alpha,
        w_cal=w_cal_local
    )
    res_p = coverage_and_width(y_test, lo_p, hi_p)
    print("\n[Point split conformal]")
    print("q:", float(q_p))
    print(res_p)

    # --- Base interval (no conformal) ---
    res_base = coverage_and_width(y_test, lo_test, hi_test)
    print("\n[Base interval]")
    print(res_base)

    # --- Base interval + split conformal expansion ---
    scores = np.maximum(lo_cal - y_cal, y_cal - hi_cal)
    scores = np.maximum(scores, 0.0)
    q_int = weighted_quantile(scores, 1 - alpha, sample_weight=w_cal_local)

    lo_conf = lo_test - q_int
    hi_conf = hi_test + q_int
    res_conf = coverage_and_width(y_test, lo_conf, hi_conf)

    print("\n[Base + conformal expansion]")
    print("q_int:", float(q_int))
    print(res_conf)

    # --- Scaled split conformal on point forecasts ---
    # score = |y - yhat| / scale
    denom = np.maximum(scale_cal, 1e-6)
    scores_s = np.abs(y_cal - yhat_cal) / denom
    q_scaled = weighted_quantile(scores_s, 1 - alpha, sample_weight=w_cal_local)

    lo_s = yhat_test - q_scaled * scale_test
    hi_s = yhat_test + q_scaled * scale_test
    res_scaled = coverage_and_width(y_test, lo_s, hi_s)

    print("\n[Scaled point split conformal]")
    print("q_scaled:", float(q_scaled))
    print(res_scaled)

    # --- Online conformal on test stream (point forecasts) ---
    step = 0.01
    out = online_conformal_point(
        y_stream=y_test,
        yhat_stream=yhat_test,
        alpha=alpha,
        step=step
    )
    print("\n[Online conformal (point, test stream)]")
    print({"coverage": out["coverage"], "avg_width": out["avg_width"], "final_q": float(out["q_series"][-1])})

    out_roll = online_conformal_point_rolling(
            y_stream=y_test,
            yhat_stream=yhat_test,
            alpha=alpha,
            window=672  # 7 days of 15-min data
        )

    print("\n[Online conformal (rolling quantile)]")
    print({
            "coverage": out_roll["coverage"],
            "avg_width": out_roll["avg_width"],
        })

Repo root added to sys.path: c:\Users\jiaya\OneDrive\Documents\Lund_2025\Thesis\unified-probabilistic-validation
Loading from: c:\Users\jiaya\OneDrive\Documents\Lund_2025\Thesis\unified-probabilistic-validation\data\derived_long
Loaded shapes: (8478,) (8478,) (8478,)
cal n: 5934 test n: 2544
w_cal min/max: 0.8801469179439237 1.1244953924308498

ALPHA=0.2  (nominal coverage=80%)

[Point split conformal]
q: 2555.0668089999963
{'n': 2544, 'coverage': 0.6544811320754716, 'avg_width': 5110.133617999993, 'median_width': 5110.133617999993}

[Base interval]
{'n': 2544, 'coverage': 0.7323113207547169, 'avg_width': 4019.2807064219737, 'median_width': 3276.19566175}

[Base + conformal expansion]
q_int: 194.02263800000947
{'n': 2544, 'coverage': 0.7963836477987422, 'avg_width': 4407.325982421992, 'median_width': 3664.240937750019}

[Scaled point split conformal]
q_scaled: 2.3450508610355025
{'n': 2544, 'coverage': 0.7861635220125787, 'avg_width': 7423.789979605287, 'median_width': 5812.49196344127

In [4]:
err_cal = np.abs(y_cal - yhat_cal)
err_test = np.abs(y_test - yhat_test)

print("Median abs error cal:", np.median(err_cal))
print("Median abs error test:", np.median(err_test))

print("Mean abs error cal:", np.mean(err_cal))
print("Mean abs error test:", np.mean(err_test))

Median abs error cal: 1356.3697434999995
Median abs error test: 1620.3224659999942
Mean abs error cal: 1603.9160268737783
Mean abs error test: 2331.524120789308


# 04 – Conformal Wrapping of Residual-Based Predictive Intervals

## Role of this Notebook

This notebook is a methodological workbench. It establishes the conformal
wrapping layer on a 90-day ENTSOG development sample before the method is
applied within the full validation pipeline. The notebook answers the question:
*which conformal variant best restores nominal coverage on a nonstationary
energy time series, and at what cost in interval width?*

The findings here inform the architectural choice made in the full pipeline
(base interval + conformal expansion) but are not themselves the official
results. Official results are produced by `experiments/run_001_entsog.py`
through `run_004_simulation.py` using the full derived datasets.

---

## Experimental Setup

**Dataset:** 90-day ENTSOG development sample (`data/derived_long/`),
15-minute resolution, n = 8,478 observations.

**Note on dataset choice.** The conformal feasibility study intentionally
uses the 90-day dev sample rather than `derived_full` (n = 209,555). This
follows the standard practice of separating methodological exploration from
official evaluation: the dev sample is used to select and justify the conformal
variant; the full dataset is used for governance classification. Using the
same dataset for both method selection and evaluation would conflate
exploratory analysis with confirmatory results.

- Calibration set: first 70% (n = 5,934)
- Test set: final 30% (n = 2,544)
- Nominal levels: 80% (α = 0.2) and 90% (α = 0.1)
- Base predictive intervals constructed via 4 time-of-day buckets,
  local window Wb = 40, global window Wg = 120, bias correction,
  residual shrinkage

**Distribution shift confirmed.** Out-of-sample error increased materially
relative to calibration, confirming nonstationarity and justifying conformal
recalibration:

| Split       | Median \|error\| | Mean \|error\| |
|-------------|-----------------|----------------|
| Calibration | 1,356           | 1,604          |
| Test        | 1,620           | 2,332          |

---

## Results: 80% Nominal Coverage (α = 0.2)

| Method                       | Coverage  | Avg Width | Median Width |
|------------------------------|-----------|-----------|--------------|
| Point Split Conformal        | 0.654     | 5,110     | 5,110        |
| Base Interval                | 0.732     | 4,019     | 3,276        |
| Base + Conformal Expansion   | **0.796** | 4,407     | 3,664        |
| Scaled Point Split Conformal | 0.786     | 7,424     | 5,812        |
| Online CP (Step Update)      | 0.499     | 3,235     | –            |
| Online CP (Rolling Quantile) | 0.833     | 8,612     | –            |

### Interpretation

**Base + conformal expansion** achieves the closest coverage to the 80%
nominal target (79.6%), with a residual shortfall of −0.4 pp. This is within
expected finite-sample tolerance for a test set of n = 2,544 and does not
indicate a structural failure of the method.

Point split conformal severely undercovers (65.4%) because it uses a
constant-width interval constructed from calibration-set residuals — when
the test distribution shifts (as confirmed by the error increase above),
a fixed quantile q cannot adapt to the new scale.

**Online CP step update** collapses to ~50% coverage at both nominal levels.
The cause is scale mismatch: the step size of 0.01 is calibrated for
normalised residuals (O(1)), but ENTSOG errors are in physical load units
(O(1,000)). The quantile update `q ← q ± step` moves at 0.01 units per
step while the true scale requires adjustments of hundreds of units — the
algorithm cannot adapt and the interval effectively stagnates. This confirms
that online CP step size must be set relative to the residual scale of the
target series.

**Online CP rolling quantile** restores coverage (83.3%) but at the cost of
substantially wider intervals (avg width 8,612 vs 4,407 for conformal
expansion), making it the least sharp of the calibrated methods.

---

## Results: 90% Nominal Coverage (α = 0.1)

| Method                       | Coverage  | Avg Width | Median Width |
|------------------------------|-----------|-----------|--------------|
| Point Split Conformal        | 0.741     | 6,573     | 6,573        |
| Base Interval                | 0.825     | 4,868     | 3,935        |
| Base + Conformal Expansion   | **0.899** | 5,443     | 4,510        |
| Scaled Point Split Conformal | 0.887     | 9,594     | 7,511        |
| Online CP (Step Update)      | 0.498     | 3,233     | –            |
| Online CP (Rolling Quantile) | 0.833     | 8,612     | –            |

### Interpretation

The structural pattern at α = 0.1 mirrors α = 0.2. Base + conformal expansion
achieves near-nominal 90% coverage (89.9%), the closest of any method, with
a width increase of only 12% over the base interval (5,443 vs 4,868). The
online CP step update failure is identical in character to the α = 0.2 case
(coverage ≈ 50%), reinforcing the scale-mismatch diagnosis. The rolling
quantile method produces the same average width as at α = 0.2 (8,612),
confirming its output is dominated by the rolling window size rather than
the nominal level.

---

## Key Findings

1. **Distribution shift is present** — out-of-sample error increases by 19%
   (median) to 45% (mean) relative to calibration, confirming nonstationarity.
2. **Constant-width split conformal fails under drift** — the fixed quantile
   cannot adapt when the test distribution shifts away from calibration.
3. **Base residual intervals under-cover** — confirming the pre-conformal
   miscalibration identified in `run_001_entsog.py`.
4. **Base + conformal expansion restores near-nominal coverage** at both
   levels with the smallest width increase of any calibrated method.
5. **Online CP step update is unstable at energy-market scale** — step=0.01
   is inappropriate for series with O(1,000) residuals; the algorithm cannot
   adapt and produces ~50% coverage regardless of nominal level.
6. **Online CP rolling quantile adapts but is not sharp** — coverage is
   restored at the cost of approximately 2× the interval width of conformal
   expansion.

---

## Conclusion

Among all evaluated methods, **base interval + conformal expansion** provides
the best calibration–sharpness trade-off and is selected as the conformal
layer for the unified validation architecture.

It:
- Achieves near-nominal coverage at both 80% and 90% levels
- Preserves the structural conditioning of the base residual intervals
- Avoids excessive width inflation (+12% at α = 0.1)
- Remains stable under distribution shift

Conformal prediction is integrated not as an additional evaluation metric but
as a calibration layer within the validation architecture. It enables a
controlled post-hoc adjustment of predictive uncertainty, allowing the
traffic-light system to distinguish between structural miscalibration
(which conformal cannot fully correct) and correctable distributional drift
(which base + conformal expansion addresses with finite-sample coverage
guarantees).